# 03 - Tool Use

Tool definitions, tool calling, parameter parsing, error handling, and tool-augmented generation.


In [ ]:
import numpy as np
import re
import math
import json
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. Tool Definitions


In [ ]:
class Tool:
    def __init__(self, name, description, parameters):
        self.name = name
        self.description = description
        self.parameters = parameters
    def validate(self, args):
        for param, ptype in self.parameters.items():
            if param not in args:
                return False, f'Missing parameter: {param}'
            if ptype == 'number' and not isinstance(args[param], (int, float)):
                return False, f'{param} must be a number'
            if ptype == 'string' and not isinstance(args[param], str):
                return False, f'{param} must be a string'
        return True, 'Valid'
    def execute(self, args):
        raise NotImplementedError

class CalculatorTool(Tool):
    def __init__(self):
        super().__init__('calculator', 'Perform calculations', {'expression': 'string'})
    def execute(self, args):
        try:
            expr = args['expression']
            allowed = {'sqrt': math.sqrt, 'log': math.log, 'exp': math.exp, 'pi': math.pi}
            result = eval(expr, {'__builtins__': {}}, allowed)
            return {'success': True, 'result': result}
        except Exception as e:
            return {'success': False, 'error': str(e)}

class WeatherTool(Tool):
    def __init__(self):
        super().__init__('weather', 'Get weather info', {'city': 'string'})
    def execute(self, args):
        city = args['city'].lower()
        mock_data = {'london': {'temp': 15, 'condition': 'Rainy'}, 'paris': {'temp': 20, 'condition': 'Sunny'}, 'tokyo': {'temp': 25, 'condition': 'Cloudy'}}
        data = mock_data.get(city, {'temp': 22, 'condition': 'Unknown'})
        return {'success': True, 'result': data}

class SearchTool(Tool):
    def __init__(self, corpus):
        super().__init__('search', 'Search documents', {'query': 'string', 'top_k': 'number'})
        self.corpus = corpus
    def execute(self, args):
        query = args['query'].lower()
        top_k = args.get('top_k', 3)
        scores = [(i, sum(1 for w in query.split() if w in doc.lower())) for i, doc in enumerate(self.corpus)]
        scores.sort(key=lambda x: x[1], reverse=True)
        return {'success': True, 'result': [{'index': i, 'text': self.corpus[i], 'score': s} for i, s in scores[:top_k]]}

corpus = ['Machine learning is amazing', 'Deep learning uses neural networks', 'NLP processes human language']
tools = {'calculator': CalculatorTool(), 'weather': WeatherTool(), 'search': SearchTool(corpus)}
print(f'Tools defined: {list(tools.keys())}')


## 2. Tool Calling


In [ ]:
class ToolCaller:
    def __init__(self, tools):
        self.tools = tools
    def call(self, tool_name, args):
        if tool_name not in self.tools:
            return {'success': False, 'error': f'Tool {tool_name} not found'}
        tool = self.tools[tool_name]
        valid, msg = tool.validate(args)
        if not valid:
            return {'success': False, 'error': msg}
        return tool.execute(args)
    def parse_call(self, text):
        try:
            match = re.search(r'\{.*?"tool"\s*:\s*"(.*?)".*?"args"\s*:\s*(\{.*?\})', text, re.DOTALL)
            if match:
                tool_name = match.group(1)
                args = json.loads(match.group(2))
                return tool_name, args
        except:
            pass
        return None, None

caller = ToolCaller(tools)

result = caller.call('calculator', {'expression': 'sqrt(144) + 5'})
print(f'Calculator result: {result}')

result2 = caller.call('weather', {'city': 'Paris'})
print(f'Weather result: {result2}')


## 3. Parameter Parsing


In [ ]:
def parse_tool_request(text):
    text_lower = text.lower()
    if 'calculate' in text_lower or 'compute' in text_lower or 'sqrt' in text_lower:
        expr_match = re.search(r'(?:calculate|compute)\s+(.*)', text, re.IGNORECASE)
        if expr_match:
            return 'calculator', {'expression': expr_match.group(1).strip()}
    if 'weather' in text_lower or 'temperature' in text_lower:
        city_match = re.search(r'(?:in|for|at)\s+([A-Za-z]+)', text)
        if city_match:
            return 'weather', {'city': city_match.group(1)}
    if 'search' in text_lower or 'find' in text_lower:
        query_match = re.search(r'(?:search|find)\s+(?:for\s+)?(.*)', text, re.IGNORECASE)
        if query_match:
            return 'search', {'query': query_match.group(1).strip(), 'top_k': 3}
    return None, None

requests = [
    'Calculate sqrt(256) divided by 4',
    'What is the weather in London?',
    'Search for machine learning papers'
]
for req in requests:
    tool, args = parse_tool_request(req)
    print(f'Request: {req}')
    print(f'  -> Tool: {tool}, Args: {args}')
    if tool:
        result = caller.call(tool, args)
        print(f'  -> Result: {result}')
    print()


## 4. Error Handling


In [ ]:
error_cases = [
    ('calculator', {'expression': '1/0'}),
    ('calculator', {'wrong_param': 'test'}),
    ('unknown_tool', {'x': 1}),
    ('weather', {'city': 123}),
]
for tool_name, args in error_cases:
    result = caller.call(tool_name, args)
    print(f'Tool: {tool_name}, Args: {args}')
    print(f'  Result: {result}')
    print()


## 5. Tool-Augmented Generation


In [ ]:
class ToolAugmentedGenerator:
    def __init__(self, tools):
        self.caller = ToolCaller(tools)
    def generate(self, query):
        tool_name, args = parse_tool_request(query)
        if tool_name:
            tool_result = self.caller.call(tool_name, args)
            if tool_result['success']:
                return self._format_response(query, tool_name, tool_result['result'])
            else:
                return f'Error using {tool_name}: {tool_result["error"]}'
        return f'No tool needed for: {query}'
    def _format_response(self, query, tool_name, result):
        if tool_name == 'calculator':
            return f'The result of your calculation is {result}'
        elif tool_name == 'weather':
            return f'Weather in {result.get("city", "the requested city")}: {result["condition"]}, {result["temp"]}C'
        elif tool_name == 'search':
            docs = result
            return f'Found {len(docs)} results: ' + '; '.join([f'{d["text"]} (score: {d["score"]})' for d in docs])
        return str(result)

generator = ToolAugmentedGenerator(tools)
queries = [
    'Calculate 15 * 23',
    'What is the weather in Tokyo?',
    'Search for deep learning'
]
for q in queries:
    print(f'Q: {q}')
    print(f'A: {generator.generate(q)}\n')


## 6. Tool Usage Statistics


In [ ]:
usage_stats = Counter()
success_stats = Counter()

def tracked_call(caller, tool_name, args):
    usage_stats[tool_name] += 1
    result = caller.call(tool_name, args)
    success_stats[tool_name] += 1 if result.get('success') else 0
    return result

for _ in range(10):
    tracked_call(caller, 'calculator', {'expression': '2 + 2'})
    tracked_call(caller, 'weather', {'city': 'Paris'})
    tracked_call(caller, 'search', {'query': 'test', 'top_k': 2})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tools_list = list(usage_stats.keys())
axes[0].bar(tools_list, [usage_stats[t] for t in tools_list], color='steelblue')
axes[0].set_title('Tool Usage Count')
axes[0].set_ylabel('Calls')
success_rates = [success_stats[t] / usage_stats[t] * 100 for t in tools_list]
axes[1].bar(tools_list, success_rates, color='green')
axes[1].set_title('Tool Success Rate')
axes[1].set_ylabel('Success %')
axes[1].set_ylim(0, 110)
plt.tight_layout()
plt.show()


## 7. Exercises


In [ ]:
# Exercise 1: Add a translation tool
# Exercise 2: Implement tool chaining
# Exercise 3: Add async tool execution
# Exercise 4: Build a tool registry with discovery
print('Exercises listed!')
